# Theo pipeline parameter search — results analysis

Scans every finished run across the result tree(s) listed in `SOURCES`
(Setup cell below), builds a tidy table of R² (and wall-clock runtime) per
(run, model, PC), and plots parameter effects per model. Nothing about
which hyperparameters exist or what their defaults are is hardcoded here --
it's read straight from `theo_pipeline_new.py`'s own argparse definitions
and from what the actual completed runs contain, so it adjusts
automatically as the search grid changes (new sweep values, new flags, a
new animal/session).

**GLC7189_c_g0 combines PC1-5, PC6-15, and PC1-15 sweeps into one view**:
WF/WC/SVR/XGB's PC1-5 and PC6-15 sweeps are combined for full PC1-15
coverage (each PC is fit completely independently for these four, via
`MultiOutputRegressor`, so the two sweeps together are equivalent to one
PC1-15 sweep); FNN/RNN/GRU/LSTM are read only from a dedicated PC1-15
*joint* sweep (these four share one hidden representation across every PC
at once, so they can't be split this way). All of this now lives in one
results/ tree -- see the comment on `SOURCES` below.

**Re-run this notebook any time** to pick up newly-finished runs -- it only
reads small `run_params.json` / `cv_summary.json` / log files, not the big
per-run pickles/models, so it stays fast no matter how large the search
directory gets. **Needs the `decoding` conda env** (not just pandas +
matplotlib): it imports `theo_pipeline_new.py` itself to read its real
argparse defaults, which pulls in torch/skorch/xgboost/spikeinterface.

## Held-out test set

Every run reserves the **last 20% of samples in time as a holdout**
(`--holdout-frac 0.2`, excluded from the CV folds entirely). But
**`evaluate_holdout` is `False` on every run in this search** (checked below
across all completed runs) -- no R² has ever been computed on that held-out
data. It was set aside but never scored. Every R² in this notebook is a
cross-validated (in-search) estimate only.

## What `oof_r2` actually is

Traced through `theo_pipeline_new.py`: for each model, `kf.split(...)`
produces `n_splits` folds over the **search portion only** (holdout excluded
before this even starts). Each fold's test predictions get written into
`oof_pred[test_idx]` -- so after all folds, `oof_pred` holds exactly one
out-of-fold prediction for every sample in the search portion, pooled
together. Then:

```python
overall = r2_score(y_search, oof_pred, multioutput="raw_values")   # oof_r2_per_output
```

**One R² computed once over the entire concatenated search-portion trace**
(not an average of per-fold R²s -- that's the separate, noisier
`mean_fold_r2` also in the table). Every sample contributes to the number
exactly once, from whichever fold held it out. This is standard "OOF R²" and
is why it's used as the primary test-set metric here: `mean_fold_r2` is the
mean of 5-10 R²s each computed on a small held-out chunk, and a couple of
unlucky small/hard chunks can swing that mean deeply negative even when the
model is fine overall (concrete example in the coverage section below).

Still **not** the same as holdout R² -- `oof_pred` never touches the reserved
20% at the end of the recording.

## Setup

In [1]:
import io
import json
import re
import sys
import contextlib
import importlib.util
from datetime import datetime
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

THEO_DECODING_DIR = Path("/raven/u/gongt/scripts/theo_decoding")
GLC6962_SEARCH_ROOT = Path("/nexus/posix0/MBR-neuralsystems/ephys/GLC6962/GLC6962_g_g0/theo_pipeline_params_search")
GLC7189_SEARCH_ROOT = Path("/nexus/posix0/MBR-neuralsystems/ephys/GLC7189/GLC7189_c_g0/theo_pipeline_params_search")  # the one shared results/ tree for GLC7189_c_g0, all PC ranges

# Which result tree(s) to read, and which (model, pc_cols) combinations to
# take from each -- a run is included only if allowed_fn(model, pc_cols)
# is True. build_search_grid.py writes every PC range (PC1-5, PC6-15,
# PC1-15) into GLC7189_SEARCH_ROOT's results/ directly -- its dedup match
# includes --pc-cols (see PIPELINE_DEFAULTS there), so a PC6-15 row is never
# confused with an already-done PC1-5 row of the same hyperparameters.
#
# WF/WC/SVR/XGB are each wrapped in sklearn's MultiOutputRegressor (see
# theo_pipeline_new.py's build_models()), which fits a completely
# independent single-output model per PC, so ANY PC range is valid for
# these four and their PC1-5 + PC6-15 sweeps are combined for full PC1-15
# coverage. FNN/RNN/GRU/LSTM are genuinely joint multi-output networks (one
# shared hidden representation feeding every PC at once), so a PC1-5-only
# fit is NOT equivalent to part of a PC1-15 fit -- those four are counted
# ONLY when a run's own pc_cols has all 15 PCs (the dedicated joint sweep),
# not just by model name, or their now-superseded PC1-5-only rows would
# silently get mixed back in.
JOINT_MODELS = {"FNN", "RNN", "GRU", "LSTM"}

def _main_source_allowed(model, n_pcs):
    return n_pcs == 15 if model in JOINT_MODELS else True

SOURCES = [
    (GLC7189_SEARCH_ROOT, _main_source_allowed),
]
PLOTS_DIR = GLC7189_SEARCH_ROOT / "analysis_plots"
PLOTS_DIR.mkdir(exist_ok=True)

# Real argparse defaults straight from theo_pipeline_new.py -- used to
# backfill any cli_args key a given run predates (e.g. an old run has no
# "spike_thresh" key at all if --spike-thresh didn't exist yet), so it reads
# as that flag's real old default instead of None/NaN. Single source of
# truth: nothing here is hand-duplicated, so it can't silently drift from
# the pipeline the way a hand-copied defaults dict would.
_spec = importlib.util.spec_from_file_location("theo_pipeline_new", THEO_DECODING_DIR / "theo_pipeline_new.py")
_tpn = importlib.util.module_from_spec(_spec)
with contextlib.redirect_stdout(io.StringIO()):
    _spec.loader.exec_module(_tpn)
ARGPARSE_DEFAULTS = {a.dest: a.default for a in _tpn.build_parser()._actions if a.dest != "help"}

# Dataset/administrative plumbing -- never a parameter-search axis, so it's
# excluded from "what did I vary" detection below. Everything else in
# cli_args is picked up automatically: add a new --foo-bar sweep knob to
# theo_pipeline_new.py / build_search_grid.py and this notebook finds it
# with no changes needed here.
#
# bin_after is also excluded even though it's a real, varying flag: this
# search always sweeps the lag window symmetrically (--bin-before ==
# --bin-after, see build_search_grid.py), so it carries no information
# beyond bin_before and would otherwise show up as a redundant, perfectly-
# correlated second axis.
ADMIN_COLS = {
    "recording_path", "peaks_path", "motion_path", "localized_peaks_path",
    "behavior_df_path", "metadata_df_path", "output_root", "animal",
    "neural_prefix", "pc_cols", "models", "seed", "device",
    "evaluate_holdout", "holdout_frac", "skip_drift_correction", "bin_after",
}
HP_COLS = [c for c in ARGPARSE_DEFAULTS if c not in ADMIN_COLS]

R2_YLIM = (-0.2, 1.0)      # fixed range for every R2 row -- clips at the edge, never autoscales to outliers
RUNTIME_YLIM = (0, 120)    # fixed range for every runtime row, in minutes -- same clipping convention
# (metric column, y-limits, row label, "one line per PC?" -- runtime is a
# per-RUN quantity, identical across a run's PCs, so it gets one line, not
# five overlapping identical ones)
METRIC_ROWS = [
    ("oof_r2", R2_YLIM, "OOF test R2", True),
    ("mean_fold_train_r2", R2_YLIM, "train R2", True),
    ("runtime_min", RUNTIME_YLIM, "runtime (min)", False),
]
MODEL_DISPLAY_ORDER = ["WF", "WC", "SVR", "XGB", "FNN", "RNN", "GRU", "LSTM"]  # cosmetic only

print("Sources (root, filter):")
for root, allowed_fn in SOURCES:
    print(f"  {root}  ->  {allowed_fn.__name__}")
print(f"{len(HP_COLS)} candidate hyperparameter/preprocessing columns tracked:")
print(sorted(HP_COLS))

/u/gongt/conda-envs/decoding/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sources (root, filter):
  /nexus/posix0/MBR-neuralsystems/ephys/GLC7189/GLC7189_c_g0/theo_pipeline_params_search  ->  _main_source_allowed
32 candidate hyperparameter/preprocessing columns tracked:
['bin_before', 'bin_current', 'channel_smooth_overlap', 'channel_smooth_window', 'dt', 'fnn_batch_size', 'fnn_frac_dropout', 'fnn_max_epochs', 'fnn_n_layers', 'fnn_num_units', 'gru_frac_dropout', 'gru_max_epochs', 'gru_n_layers', 'gru_num_units', 'input_preprocess', 'lstm_frac_dropout', 'lstm_max_epochs', 'lstm_n_layers', 'lstm_num_units', 'n_splits', 'output_preprocess', 'rnn_frac_dropout', 'rnn_max_epochs', 'rnn_num_units', 'spike_thresh', 'svr_c', 'svr_max_iter', 'wc_degree', 'wf_alphas', 'xgb_learning_rate', 'xgb_max_depth', 'xgb_n_estimators']


## Load every finished run into one tidy table (one row per run x model x PC), plus per-run wall-clock time

In [2]:
def scan_runtimes(search_root):
    '''Match each run to its log file via the "Saved run params -> ..." line
    the pipeline prints, then pull Start time / End time from that same log.'''
    run_time = {}
    for f in (search_root / "logs").glob("*.log"):
        text = f.read_text(errors="replace")
        m = re.search(r"^Saved run params\s+-> (.+)/run_params\.json", text, re.M)
        if not m:
            continue
        run_dir = Path(m.group(1)).name
        start_m = re.search(r"^Start time: (\S+)", text, re.M)
        end_m = re.search(r"^End time: (\S+)", text, re.M)
        if start_m and end_m:
            start = datetime.fromisoformat(start_m.group(1))
            end = datetime.fromisoformat(end_m.group(1))
            run_time[run_dir] = (end - start).total_seconds()
    return run_time

rows = []
incomplete_runs = []
n_runtime_matched = 0
for root, allowed_fn in SOURCES:
    results_root = root / "results"
    runtime_by_run = scan_runtimes(root)
    n_runtime_matched += len(runtime_by_run)

    for run_dir in sorted(results_root.glob("[0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9]_*")):
        if not run_dir.is_dir():
            continue
        params_f, summary_f = run_dir / "run_params.json", run_dir / "cv_summary.json"
        if not params_f.exists() or not summary_f.exists():
            incomplete_runs.append(run_dir.name)
            continue
        params = json.loads(params_f.read_text())
        summary = json.loads(summary_f.read_text())
        cli = params["cli_args"]
        pc_cols = params["resolved"]["pc_cols"]
        runtime_sec = runtime_by_run.get(run_dir.name)

        # Every knob theo_pipeline_new.py exposes, except pure plumbing (ADMIN_COLS)
        # -- backfilled to its real argparse default when this run predates that
        # flag entirely, so old runs read as "at the old default", not None/NaN
        # (which would silently drop them out of every plot filtered to that
        # default).
        hp_vals = {c: cli.get(c, ARGPARSE_DEFAULTS[c]) for c in HP_COLS}

        for model, m in summary.get("models", {}).items():
            if not allowed_fn(model, len(pc_cols)):
                continue
            oof = m.get("oof_r2_per_output")
            mean_fold = m.get("mean_fold_r2_per_output")
            mean_fold_train = m.get("mean_fold_train_r2_per_output")
            fold_raw = m.get("fold_r2_raw_values")
            for i, pc in enumerate(pc_cols):
                row = {
                    "run_dir": run_dir.name,
                    "run_root": str(root),  # which SOURCES entry this run lives under -- decoded_trace_png needs it
                    "timestamp": params.get("timestamp"),
                    "model": model,
                    "pc": pc,
                    "oof_r2": oof[i] if oof else np.nan,
                    "mean_fold_r2": mean_fold[i] if mean_fold else np.nan,
                    "mean_fold_train_r2": mean_fold_train[i] if mean_fold_train else np.nan,
                    "fold_r2_std": float(np.std([f[i] for f in fold_raw])) if fold_raw else np.nan,
                    "runtime_sec": runtime_sec,
                    "runtime_min": runtime_sec / 60 if runtime_sec is not None else np.nan,
                    "evaluate_holdout": cli.get("evaluate_holdout", False),
                }
                row.update(hp_vals)
                rows.append(row)

print(f"{n_runtime_matched} runs (across all sources) matched to a log with start+end time.")
df = pd.DataFrame(rows)
df.to_csv(PLOTS_DIR / "all_runs_long.csv", index=False)

ALL_PCS = sorted(df["pc"].unique(), key=lambda p: int(p[2:]))  # numeric, not lexicographic (PC2 < PC10)
PC_COLORS = {pc: c for pc, c in zip(ALL_PCS, plt.cm.tab10(np.linspace(0, 1, max(len(ALL_PCS), 1))))}
ALL_MODELS = sorted(df["model"].unique(),
                     key=lambda m: (MODEL_DISPLAY_ORDER.index(m) if m in MODEL_DISPLAY_ORDER else 99, m))

print(f"{df['run_dir'].nunique()} completed runs loaded ({len(df)} run x model x PC rows).")
print(f"{len(incomplete_runs)} run directories are still in progress or permanently failed (no cv_summary.json) -- skipped.")
print(f"PCs found: {ALL_PCS}")
print()
print("Completed runs per model:")
print(df["model"].value_counts().reindex(ALL_MODELS).to_string())
print()
print(f"{df['runtime_sec'].isna().sum()} rows missing a runtime match (should be 0).")
assert not df["evaluate_holdout"].any(), "evaluate_holdout is no longer always False -- update the markdown note above"
print("Confirmed: evaluate_holdout is False on every completed run.")

6922 runs (across all sources) matched to a log with start+end time.
4525 completed runs loaded (60405 run x model x PC rows).
170 run directories are still in progress or permanently failed (no cv_summary.json) -- skipped.
PCs found: ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'PC11', 'PC12', 'PC13', 'PC14', 'PC15']

Completed runs per model:
model
WF       1180
WC        810
SVR       980
XGB      1755
FNN     10005
RNN      9825
GRU     17925
LSTM    17925

30 rows missing a runtime match (should be 0).
Confirmed: evaluate_holdout is False on every completed run.


## Coverage — what's actually been tested per model

`oof_r2` vs `mean_fold_r2` for the same run (why this notebook standardizes
on `oof_r2`):

In [3]:
example = df[df.model == "SVR"].iloc[:5][["run_dir", "pc", "oof_r2", "mean_fold_r2"]]
if not example.empty:
    print(example.to_string(index=False))
    print()
    print("-> mean_fold_r2 swings far more negative: a couple of small, hard test folds")
    print("   dominate the mean-of-per-fold-R2 metric. oof_r2 (one R2 over the pooled")
    print("   out-of-fold predictions) is the steadier estimate -- see markdown above.")

                        run_dir  pc    oof_r2  mean_fold_r2
20260808_023103_SVR_c0.3_mi1000 PC1 -0.132561    -48.985811
20260808_023103_SVR_c0.3_mi1000 PC2 -0.170440    -10.145121
20260808_023103_SVR_c0.3_mi1000 PC3 -0.154708    -11.858799
20260808_023103_SVR_c0.3_mi1000 PC4 -0.432187    -11.006199
20260808_023103_SVR_c0.3_mi1000 PC5 -0.048891     -6.285437

-> mean_fold_r2 swings far more negative: a couple of small, hard test folds
   dominate the mean-of-per-fold-R2 metric. oof_r2 (one R2 over the pooled
   out-of-fold predictions) is the steadier estimate -- see markdown above.


In [4]:
def varying_cols(sub):
    '''Columns among HP_COLS with >1 distinct value within this subset --
    auto-detects exactly what was actually varied for this model, no
    hardcoded per-model parameter list.'''
    return [c for c in HP_COLS if sub[c].nunique(dropna=True) > 1]

def anchor_value(sub, c):
    '''Value to hold column c at when plotting/detecting a grid over some
    OTHER varying column(s) -- prefers theo_pipeline_new.py's real argparse
    default when it's actually present in this subset's domain (it almost
    always is: everything except a sweep that specifically targets c stays
    at the pipeline default for c), falling back to the statistical mode
    only if that default was never run at all for this model.

    Plain "most common value" alone isn't reliable here: e.g. bin_before=6
    is theo_pipeline_new.py's default and what every model's OWN
    hyperparameter grid runs at, but the separate dt x bin_before x
    n_splits pipeline sweep spreads its rows evenly across 6 *other* window
    values -- each of those individually outnumbers the 6-default rows, so
    the raw mode of bin_before comes out as one of the swept values, not 6,
    which would anchor every other model-hyperparameter heatmap/marginal at
    a window size its own grid was never actually run at.'''
    vals = sub[c].dropna()
    default = ARGPARSE_DEFAULTS.get(c)
    if default is not None and not vals.empty and _match(vals, default).any():
        return default
    return vals.mode().iloc[0] if not vals.empty else default

def coverage_report(df):
    recs = []
    for model in ALL_MODELS:
        sub = df[df["model"] == model]
        runs = sub.drop_duplicates("run_dir")
        varying = varying_cols(sub)
        if not varying:
            summary = "(nothing varies -- only ever run at its defaults)"
        else:
            summary = "; ".join(f"{c}={sorted(runs[c].dropna().unique().tolist())}" for c in varying)
        recs.append({"model": model, "n_runs": len(runs), "varied": summary})
    return pd.DataFrame(recs)

pd.set_option("display.max_colwidth", 200)
cov = coverage_report(df)
cov.to_csv(PLOTS_DIR / "coverage_report.csv", index=False)
print(cov.to_string(index=False))

model  n_runs                                                                                                                                                                                                                                                                                                                               varied
   WF     189                                                                                                                                                                     dt=[0.1, 0.2, 0.5, 1.0, 2.0, 4.0]; bin_before=[1, 3, 5, 6, 10, 15, 20, 25, 30, 35]; n_splits=[5, 10, 15, 20, 25, 30, 35, 40]; spike_thresh=[0, 300, 5000, 20000]
   WC     157                                                                                                                                                                         dt=[0.1, 0.2, 0.5, 1.0, 2.0, 4.0]; bin_before=[1, 3, 5, 6, 10, 15, 20]; n_splits=[5, 10, 15, 20]; spike_thresh=[0, 300, 5000, 20000]; wc_deg

## Plot conventions used throughout

Nothing below is told in advance which hyperparameters a model has or what
was swept -- for each model, any column with more than one distinct value
among its completed runs (see coverage above) counts as "varied" and gets
plotted automatically.

- **Marginals** (`{model}_marginals.png`): one PNG per model, one column per
  varied parameter, **3 stacked rows** (OOF test R², train R², runtime).
  Every other varied parameter is held at its most-common ("mode") value.
  R² rows draw **one colored line per PC** (no averaging across PCs, no
  error bars) on a **fixed y-axis of (-0.2, 1)** -- a point outside that
  band is clipped at the edge, never autoscaled to outliers. The runtime
  row is a single line (runtime doesn't vary by PC).
- **Heatmaps** (`{model}_heatmap_<params>.png`): whenever two (or three)
  parameters were actually swept *together* as a real grid -- not just
  one-at-a-time -- rather than averaging over PCs, each PC gets its **own
  heatmap row**, all stacked into the same PNG (plus one shared runtime
  row at the bottom, again not split by PC). Columns are the grid's third
  swept dimension, if it has one (e.g. `n_splits`, for the `dt` x
  `bin_before` grid). A parameter is only called a "real grid" with
  another if their completed runs actually cover more than a one-at-a-time
  "+"-shaped cross -- see `detect_grid_pairs` below.

In [5]:
def _match(series, value):
    if pd.api.types.is_numeric_dtype(series):
        return np.isclose(series.astype(float), float(value))
    return series == value

def _fmt(v):
    return f"{v:g}" if isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, bool) else str(v)

def _domain_info(domain):
    '''Numeric domains keep their real spacing (so unevenly-spaced sweep
    values, e.g. spike_thresh=[300,5000,20000], still plot at their true x);
    categorical/string domains (e.g. output_preprocess) get evenly spaced
    integer slots instead.'''
    numeric = all(isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, bool) for v in domain)
    xs = sorted(domain) if numeric else sorted(domain, key=str)
    pos = {v: (v if numeric else i) for i, v in enumerate(xs)}
    return xs, pos, numeric

def _should_logx(domain):
    vals = [v for v in domain if isinstance(v, (int, float, np.integer, np.floating))
            and not isinstance(v, bool) and v > 0]
    if len(vals) < 2:
        return False
    return max(vals) / min(vals) >= 10

def _set_x_domain(ax, domain, logx):
    '''Fix x ticks/limits to the FULL set of values this parameter is ever
    tested at for this model (not just whatever's present after filtering to
    one anchor) -- so a panel with only 1 of e.g. 4 values completed still
    shows the right context instead of matplotlib auto-zooming to a single
    point and looking broken.'''
    xs, pos, numeric = _domain_info(domain)
    if len(xs) < 2:
        return pos
    tick_positions = [pos[v] for v in xs]
    ax.set_xticks(tick_positions)
    ax.set_xticklabels([_fmt(v) for v in xs])
    if logx and numeric:
        ax.set_xscale("log")
        # A value like spike_thresh=0 can't be placed on a log axis at all --
        # base the lower limit on the smallest POSITIVE tick instead of xs[0],
        # so that one non-positive value doesn't make the axis limits invalid
        # for every other (real, positive) tick.
        positive_xs = [v for v in xs if v > 0]
        lo = (positive_xs[0] / 1.3) if positive_xs else xs[0]
        hi = xs[-1] * 1.3
    else:
        margin = (tick_positions[-1] - tick_positions[0]) * 0.08 or 0.5
        lo, hi = tick_positions[0] - margin, tick_positions[-1] + margin
    ax.set_xlim(lo, hi)
    return pos

def marginal_panel(sub, x_col, metric, ax, ylim, by_pc, pos):
    '''One subplot: x_col vs metric, one colored line per PC (or a single
    line if this metric doesn't vary by PC, e.g. runtime).'''
    if ylim:
        ax.axhline(0, color="black", linewidth=0.6, alpha=0.4)
        ax.set_ylim(*ylim)
    if sub.empty or sub[metric].dropna().empty:
        ax.text(0.5, 0.5, "no data yet", ha="center", va="center", transform=ax.transAxes, fontsize=8, color="gray")
        return
    if by_pc:
        for pc in ALL_PCS:
            g = sub.loc[sub["pc"] == pc].dropna(subset=[metric]).groupby(x_col)[metric].mean()
            if g.empty:
                continue
            xs_sorted = sorted(g.index, key=lambda v: pos[v])
            ax.plot([pos[v] for v in xs_sorted], [g[v] for v in xs_sorted], "o-",
                    color=PC_COLORS[pc], label=pc, markersize=4, linewidth=1.3)
        ax.legend(fontsize=6, ncol=min(len(ALL_PCS), 3))
    else:
        g = sub.drop_duplicates("run_dir").dropna(subset=[metric]).groupby(x_col)[metric].mean()
        if g.empty:
            ax.text(0.5, 0.5, "no data yet", ha="center", va="center", transform=ax.transAxes, fontsize=8, color="gray")
            return
        xs_sorted = sorted(g.index, key=lambda v: pos[v])
        ax.plot([pos[v] for v in xs_sorted], [g[v] for v in xs_sorted], "o-", color="#3B6FA0", markersize=5, linewidth=1.5)

def detect_grid_pairs(sub, varying):
    '''Which pairs of varying columns were actually swept TOGETHER as a real
    grid (both sides taking >=2 values, jointly, with the anchor point for
    every other varying column held at its mode) -- as opposed to a plain
    one-at-a-time "+"-shaped sweep, where only one of the two ever moves off
    its default at a time. A "+"-shape with nx values on one arm and ny on
    the other populates exactly nx+ny-1 cells; a real (possibly partial)
    grid populates more than that.'''
    anchors = {c: anchor_value(sub, c) for c in varying}
    pairs = []
    for x, y in combinations(varying, 2):
        others = [c for c in varying if c not in (x, y)]
        s = sub
        for c in others:
            s = s.loc[_match(s[c], anchors[c])]
        nx, ny = s[x].nunique(), s[y].nunique()
        if nx < 2 or ny < 2:
            continue
        populated = s.drop_duplicates([x, y]).shape[0]
        if populated > nx + ny - 1:
            pairs.append((x, y))
    return pairs, anchors

def cluster_pairs(varying, pairs):
    '''Union-find over "real grid" pairs -- e.g. dt/bin_before/n_splits all
    pairwise qualify (see build_search_grid.py's pipeline_sweep_rows), so
    they merge into one 3-way cluster instead of three separate heatmaps.'''
    parent = {c: c for c in varying}
    def find(c):
        while parent[c] != c:
            c = parent[c]
        return c
    for x, y in pairs:
        rx, ry = find(x), find(y)
        if rx != ry:
            parent[rx] = ry
    clusters = {}
    for c in varying:
        clusters.setdefault(find(c), []).append(c)
    return [c for c in clusters.values() if len(c) >= 2]

def heatmap_cell(full_sub, sub, hp_x, hp_y, metric, ax, clim, cmap_name):
    all_x = sorted(full_sub[hp_x].dropna().unique(), key=str if not pd.api.types.is_numeric_dtype(full_sub[hp_x]) else None)
    all_y = sorted(full_sub[hp_y].dropna().unique(), key=str if not pd.api.types.is_numeric_dtype(full_sub[hp_y]) else None)
    ax.set_xticks(range(len(all_x))); ax.set_xticklabels([_fmt(v) for v in all_x], fontsize=7, rotation=30, ha="right")
    ax.set_yticks(range(len(all_y))); ax.set_yticklabels([_fmt(v) for v in all_y], fontsize=7)
    if sub.empty or sub[metric].dropna().empty:
        ax.text(0.5, 0.5, "no data\nyet", ha="center", va="center", fontsize=7, color="gray", transform=ax.transAxes)
        return None
    piv = sub.groupby([hp_y, hp_x])[metric].mean().unstack(hp_x)
    piv = piv.reindex(index=all_y, columns=all_x)
    masked = np.ma.masked_invalid(piv.values)
    cmap = plt.get_cmap(cmap_name).copy()
    cmap.set_bad("lightgray")
    im = ax.imshow(masked, cmap=cmap, vmin=clim[0], vmax=clim[1], aspect="auto")
    for i in range(len(all_y)):
        for j in range(len(all_x)):
            v = piv.values[i, j]
            ax.text(j, i, f"{v:.2f}" if np.isfinite(v) else "-", ha="center", va="center", fontsize=6)
    return im

def heatmap_for_cluster(model, sub, cluster, anchors, savepath):
    '''One PNG: rows = (oof_r2 x PC) + (train_r2 x PC) + one shared runtime
    row; columns = the cluster's 3rd dimension (if any). Any 4th+ dimension
    in the cluster is held fixed at its mode value (noted in the title) --
    a compromise so the figure stays 2D-navigable; that dimension still gets
    its own marginal panel in {model}_marginals.png.'''
    sizes = sorted(cluster, key=lambda c: (-sub[c].nunique(), HP_COLS.index(c)))
    hp_x, hp_y = sizes[0], sizes[1]
    facet_col = sizes[2] if len(sizes) >= 3 else None
    fixed_extra = {c: anchors[c] for c in sizes[3:]}

    facet_vals = sorted(sub[facet_col].dropna().unique(), key=str) if facet_col else [None]
    cmaps = {"oof_r2": "RdYlGn", "mean_fold_train_r2": "RdYlGn", "runtime_min": "viridis"}
    clims = {"oof_r2": R2_YLIM, "mean_fold_train_r2": R2_YLIM, "runtime_min": RUNTIME_YLIM}
    row_specs = [(metric, pc) for metric, _, _, by_pc in METRIC_ROWS for pc in (ALL_PCS if by_pc else [None])]

    ncols = len(facet_vals)
    nrows = len(row_specs)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.4 * ncols + 1.2, 2.3 * nrows), squeeze=False)

    base = sub
    for c, extra_v in fixed_extra.items():
        base = base.loc[_match(base[c], extra_v)]

    for r, (metric, pc) in enumerate(row_specs):
        ims = []
        for c, fval in enumerate(facet_vals):
            ax = axes[r][c]
            s = base if fval is None else base.loc[_match(base[facet_col], fval)]
            if pc is not None:
                s = s.loc[s["pc"] == pc]
            im = heatmap_cell(base, s, hp_x, hp_y, metric, ax, clims[metric], cmaps[metric])
            if im is not None:
                ims.append(im)
            if r == 0 and facet_col:
                ax.set_title(f"{facet_col}={_fmt(fval)}", fontsize=9)
            if c == 0:
                ax.set_ylabel(f"{metric}\n({pc})" if pc else metric, fontsize=8)
            if r == nrows - 1:
                ax.set_xlabel(hp_x, fontsize=8)
        if ims:
            fig.colorbar(ims[0], ax=axes[r].tolist(), shrink=0.85)

    extra_note = f" -- {fixed_extra} held at default" if fixed_extra else ""
    title = f"{model}: {hp_y} (rows) x {hp_x} (cols)"
    if facet_col:
        title += f", faceted by {facet_col}"
    fig.suptitle(title + extra_note, fontsize=11)
    fig.savefig(PLOTS_DIR / savepath, dpi=150, bbox_inches="tight")
    plt.show()
    return fig

## Per-model marginals and heatmaps — fully auto-detected from what was actually run

For every model: a `{model}_marginals.png` for every parameter that varies
at all, and a `{model}_heatmap_<params>.png` for every group of parameters
that were swept together as a real grid (see `detect_grid_pairs` above) --
including the shared `dt` / `bin_before` / `n_splits` pipeline sweep, which
applies the same way to every model.

In [6]:
def model_marginals_png(model):
    sub = df[df.model == model]
    varying = varying_cols(sub)
    if not varying:
        print(f"{model}: nothing varies (only ever run at its defaults) -- skipping marginals.")
        return None
    anchors = {c: anchor_value(sub, c) for c in varying}
    full_domains = {c: sorted(sub[c].dropna().unique(), key=str) for c in varying}

    nrows, ncols = len(METRIC_ROWS), len(varying)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.4 * ncols, 3.1 * nrows), squeeze=False)
    for r, (metric, ylim, mlabel, by_pc) in enumerate(METRIC_ROWS):
        for c, x_col in enumerate(varying):
            ax = axes[r][c]
            s = sub
            for oc in varying:
                if oc != x_col:
                    s = s.loc[_match(s[oc], anchors[oc])]
            pos = _set_x_domain(ax, full_domains[x_col], _should_logx(full_domains[x_col]))
            marginal_panel(s, x_col, metric, ax, ylim, by_pc, pos)
            if r == 0:
                ax.set_title(x_col, fontsize=9)
            ax.set_ylabel(mlabel if c == 0 else "")
            if r == nrows - 1:
                ax.set_xlabel(x_col, fontsize=8)
    fig.suptitle(f"{model}: one-parameter-at-a-time (each panel holds every other varied knob at its most-common value)")
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / f"{model}_marginals.png", dpi=150)
    plt.show()
    return fig

def model_heatmaps(model):
    sub = df[df.model == model]
    varying = varying_cols(sub)
    if len(varying) < 2:
        return []
    pairs, anchors = detect_grid_pairs(sub, varying)
    clusters = cluster_pairs(varying, pairs)
    if not clusters:
        print(f"{model}: no 2D grid detected yet among {varying} (only one-at-a-time sweeps so far).")
        return []
    figs = []
    for cluster in clusters:
        tag = "_".join(sorted(cluster))
        figs.append(heatmap_for_cluster(model, sub, cluster, anchors, f"{model}_heatmap_{tag}.png"))
    return figs

for model in ALL_MODELS:
    print(f"=== {model} ===")
    model_marginals_png(model)
    model_heatmaps(model)

=== WF ===


/tmp/ipykernel_122504/3134613945.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_122504/3665361052.py:189: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


=== WC ===


/tmp/ipykernel_122504/3134613945.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_122504/3665361052.py:189: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


=== SVR ===


/tmp/ipykernel_122504/3134613945.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_122504/3665361052.py:189: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_122504/3665361052.py:189: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


... (6 outputs left out)

=== GRU ===


/tmp/ipykernel_122504/3134613945.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_122504/3665361052.py:189: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_122504/3665361052.py:189: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


=== LSTM ===


/tmp/ipykernel_122504/3134613945.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_122504/3665361052.py:158: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(nrows, ncols, figsize=(3.4 * ncols + 1.2, 2.3 * nrows), squeeze=False)
/tmp/ipykernel_122504/3665361052.py:189: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_122504/3665361052.py:189: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Leaderboard — best configuration found so far, per model family

Best run selected by mean OOF test R² across PCs; its train R² and runtime
are shown for that same run (not re-optimized separately). Same 3-row layout,
one column: all models side by side within each row.

In [7]:
def leaderboard(df, metric="oof_r2"):
    recs = []
    for model in ALL_MODELS:
        sub = df[df.model == model]
        by_run = sub.groupby("run_dir")[metric].mean()
        if by_run.empty:
            continue
        best_run = by_run.idxmax()
        best = sub[sub.run_dir == best_run]
        best_hp = {c: best[c].iloc[0] for c in varying_cols(sub)}
        recs.append({
            "model": model, "best_run": best_run, "best_hyperparams": best_hp,
            "test_mean": best["oof_r2"].mean(), "test_min": best["oof_r2"].min(), "test_max": best["oof_r2"].max(),
            "train_mean": best["mean_fold_train_r2"].mean(),
            "train_min": best["mean_fold_train_r2"].min(), "train_max": best["mean_fold_train_r2"].max(),
            "runtime_min": best["runtime_min"].iloc[0],
            "n_runs_tested": sub.run_dir.nunique(),
        })
    return pd.DataFrame(recs).sort_values("test_mean", ascending=False)

lb = leaderboard(df)
lb.to_csv(PLOTS_DIR / "leaderboard.csv", index=False)
print(lb.to_string(index=False))

fig, axes = plt.subplots(3, 1, figsize=(6.5, 9.5))
x = np.arange(len(lb))
test_err = np.vstack([(lb.test_mean - lb.test_min).clip(lower=0), (lb.test_max - lb.test_mean).clip(lower=0)])
train_err = np.vstack([(lb.train_mean - lb.train_min).clip(lower=0), (lb.train_max - lb.train_mean).clip(lower=0)])
axes[0].bar(x, lb.test_mean, yerr=test_err, capsize=4, color="#3B6FA0")
axes[0].axhline(0, color="black", linewidth=0.6, alpha=0.4)
axes[0].set_ylim(*R2_YLIM); axes[0].set_ylabel("OOF test R2")
axes[1].bar(x, lb.train_mean, yerr=train_err, capsize=4, color="#E07B39")
axes[1].axhline(0, color="black", linewidth=0.6, alpha=0.4)
axes[1].set_ylim(*R2_YLIM); axes[1].set_ylabel("train R2")
axes[2].bar(x, lb.runtime_min, color="#4C9A6A")
axes[2].set_ylabel("runtime (min)")
for ax in axes:
    ax.set_xticks(x); ax.set_xticklabels(lb.model)
fig.suptitle("Leaderboard: best configuration found so far, per model")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "leaderboard.png", dpi=150)
plt.show()

model                              best_run                                                                                                                                              best_hyperparams  test_mean  test_min  test_max  train_mean  train_min  train_max  runtime_min  n_runs_tested
  GRU   20260812_045823_GRU_u400_do0.25_e10     {'dt': 4.0, 'bin_before': 40, 'n_splits': 40, 'spike_thresh': 0, 'gru_num_units': 400, 'gru_frac_dropout': 0.25, 'gru_max_epochs': 10, 'gru_n_layers': 2}   0.297052 -0.016688  0.682021    0.726750   0.367203   0.921578     1.766667           1195
   WC             20260808_171222_WC_deg3_1                                                                              {'dt': 4.0, 'bin_before': 20, 'n_splits': 20, 'spike_thresh': 0, 'wc_degree': 3}   0.266417 -0.066093  0.486718    1.000000   1.000000   1.000000     1.783333            157
   WF               20260809_230338_WF_na25                                                                        

/tmp/ipykernel_122504/3742406976.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Decoded trace — best and worst run overall

Reconstructed from each run's own saved OOF predictions
(`results/<run>/predictions/<model>_oof_predictions.npz` -- the exact same
array theo_pipeline_new.py's own `{model}_decoded_vs_observed.png` was built
from, not a re-derived approximation), one panel per PC, with the full
hyperparameter combination for that run spelled out in the title. "Best"/
"worst" = highest/lowest mean OOF R² across a run's own PCs, over every
completed run in this search (not per-model-family -- see the leaderboard
above for that).

In [8]:
def decoded_trace_png(run_dir_name, title_prefix, savepath):
    row0 = df.loc[df.run_dir == run_dir_name].iloc[0]
    run_dir = Path(row0["run_root"]) / "results" / run_dir_name  # run_root: which SOURCES entry this run came from
    model = row0["model"]
    npz_path = run_dir / "predictions" / f"{model}_oof_predictions.npz"
    if not npz_path.exists():
        print(f"No saved OOF predictions for {run_dir_name} ({npz_path}) -- skipping.")
        return None
    d = np.load(npz_path)
    t, y_true, y_pred = d["t"], d["y_true"], d["y_pred"]
    # pc_cols straight from this run's own run_params.json -- guaranteed to be
    # in the same column order as y_true/y_pred above (both written from the
    # same pc_cols list in the same pipeline run), unlike re-deriving it from
    # df (which would silently re-sort and risk mislabeling which column is
    # which PC).
    params = json.loads((run_dir / "run_params.json").read_text())
    pc_cols = params["resolved"]["pc_cols"]

    hp_str = ", ".join(f"{c}={_fmt(row0[c])}" for c in varying_cols(df[df.model == model]))

    n_out = y_true.shape[1]
    fig, axes = plt.subplots(n_out, 1, figsize=(14, 2.3 * n_out), sharex=True, squeeze=False)
    axes = axes[:, 0]
    for i, ax in enumerate(axes):
        r2 = r2_score(y_true[:, i], y_pred[:, i])
        ax.plot(t, y_true[:, i], color="black", lw=0.8, label="Observed")
        ax.plot(t, y_pred[:, i], color="tab:red", lw=0.8, alpha=0.9, label="Decoded")
        ax.set_ylabel(f"{pc_cols[i]}\n(R2={r2:.2f})")
        ax.grid(alpha=0.2)
        if i == 0:
            ax.legend(loc="upper right", fontsize=8)
    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(f"{title_prefix}\n{run_dir_name} -- {model}: {hp_str}", fontsize=10)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / savepath, dpi=130, bbox_inches="tight")
    plt.show()
    return fig

by_run_mean = df.groupby("run_dir")["oof_r2"].mean().dropna()
if by_run_mean.empty:
    print("No completed runs with a valid oof_r2 yet.")
else:
    best_run, worst_run = by_run_mean.idxmax(), by_run_mean.idxmin()
    print(f"Best run overall:  {best_run} (mean OOF R2 = {by_run_mean[best_run]:.3f})")
    print(f"Worst run overall: {worst_run} (mean OOF R2 = {by_run_mean[worst_run]:.3f})")
    decoded_trace_png(best_run, "BEST run overall (highest mean OOF R2 across its PCs)", "best_run_decoded_trace.png")
    decoded_trace_png(worst_run, "WORST run overall (lowest mean OOF R2 across its PCs)", "worst_run_decoded_trace.png")

Best run overall:  20260812_045823_GRU_u400_do0.25_e10 (mean OOF R2 = 0.297)
Worst run overall: 20260808_154443_WC_deg3 (mean OOF R2 = -1237213382255960697864192.000)


/tmp/ipykernel_122504/2716533958.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_122504/2716533958.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
